<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">

## LAB 03 — Entregável vendedor (challenge 003)

Ferramenta de priorização do **pipe aberto** (Prospecting + Engaging). Não ordena por `close_value` (é null no aberto).

1. Rode o **LAB 01** antes.
2. Widgets: `catalog_name`, `schema_name`, `sales_agent` (vazio = todos).
3. **Run All**. Cada linha traz `score` + `why`.

`GTXPro` não junta em `GTX Pro`. Conta pode ser null — left join.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "crm")
agent_filter = widget_or_default("sales_agent", "")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name} agent={agent_filter or '*'}")

### Tile 1 — Pipe aberto vs resolvido

In [ ]:
display(
    spark.sql(
        """
SELECT
  deal_stage,
  COUNT(*) AS opportunities,
  SUM(CASE WHEN account IS NULL THEN 1 ELSE 0 END) AS missing_account,
  SUM(CASE WHEN close_value IS NULL THEN 1 ELSE 0 END) AS missing_close_value
FROM sales_pipeline
GROUP BY deal_stage
ORDER BY opportunities DESC
"""
    )
)

### Tile 2 — Prioridade desta segunda (score + por quê)

Pontos: Engaging +35; conta com `revenue` acima da mediana +20; produto casa no catálogo (depois de `GTXPro`→`GTX Pro`) +15; win rate histórico do agente ≥ mediana +15; `engage_date` nos últimos 45 dias +15. Sem conta = 0 no bloco de receita (não zera o deal).

In [ ]:
safe_agent = agent_filter.replace("'", "")
agent_clause = f"AND p.sales_agent = '{safe_agent}'" if safe_agent else ""
display(
    spark.sql(
        f"""
WITH resolved AS (
  SELECT sales_agent,
         AVG(CASE WHEN deal_stage = 'Won' THEN 1.0 ELSE 0.0 END) AS agent_win_rate
  FROM sales_pipeline
  WHERE deal_stage IN ('Won', 'Lost')
  GROUP BY sales_agent
),
rev_med AS (
  SELECT percentile_approx(revenue, 0.5) AS med_revenue FROM accounts
),
win_med AS (
  SELECT percentile_approx(agent_win_rate, 0.5) AS med_win FROM resolved
)
SELECT
  p.opportunity_id,
  p.sales_agent,
  t.manager,
  t.regional_office,
  p.deal_stage,
  p.product AS pipeline_product,
  pr.product AS catalog_product,
  p.account,
  a.sector,
  a.revenue,
  p.engage_date,
  CAST(
    (CASE WHEN p.deal_stage = 'Engaging' THEN 35 ELSE 10 END)
    + (CASE WHEN a.revenue IS NOT NULL AND a.revenue >= rev_med.med_revenue THEN 20 ELSE 0 END)
    + (CASE WHEN pr.product IS NOT NULL THEN 15 ELSE 0 END)
    + (CASE WHEN r.agent_win_rate IS NOT NULL AND r.agent_win_rate >= win_med.med_win THEN 15 ELSE 0 END)
    + (CASE WHEN p.engage_date IS NOT NULL AND p.engage_date >= date_sub(current_date(), 45) THEN 15 ELSE 0 END)
  AS INT) AS score,
  concat_ws(' | ',
    CASE WHEN p.deal_stage = 'Engaging' THEN 'stage Engaging (+35)' ELSE 'stage Prospecting (+10)' END,
    CASE WHEN a.revenue IS NOT NULL AND a.revenue >= rev_med.med_revenue THEN 'conta acima da mediana de revenue (+20)'
         WHEN p.account IS NULL THEN 'sem conta — não inner-join'
         ELSE 'conta abaixo da mediana de revenue' END,
    CASE WHEN pr.product IS NOT NULL THEN 'produto no catálogo (+15)' ELSE 'órfão GTXPro/catálogo (+0)' END,
    CASE WHEN r.agent_win_rate IS NOT NULL AND r.agent_win_rate >= win_med.med_win THEN 'agente com win rate alto (+15)' ELSE 'win rate do agente abaixo da mediana' END,
    CASE WHEN p.engage_date IS NOT NULL AND p.engage_date >= date_sub(current_date(), 45) THEN 'engage recente (+15)' ELSE 'engage antigo ou nulo' END
  ) AS why
FROM sales_pipeline p
LEFT JOIN sales_teams t ON p.sales_agent = t.sales_agent
LEFT JOIN accounts a ON p.account = a.account
LEFT JOIN products pr ON replace(p.product, 'GTXPro', 'GTX Pro') = pr.product
LEFT JOIN resolved r ON p.sales_agent = r.sales_agent
CROSS JOIN rev_med
CROSS JOIN win_med
WHERE p.deal_stage IN ('Prospecting', 'Engaging')
{agent_clause}
ORDER BY score DESC, a.revenue DESC NULLS LAST
LIMIT 40
"""
    )
)

### Tile 3 — Produto órfão no pipe aberto

`GTXPro` só casa depois do replace para `GTX Pro`. O que sobrar não entra no score de catálogo.

In [ ]:
display(
    spark.sql(
        """
SELECT
  p.product AS pipeline_product,
  COUNT(*) AS open_deals
FROM sales_pipeline p
LEFT JOIN products pr
  ON replace(p.product, 'GTXPro', 'GTX Pro') = pr.product
WHERE p.deal_stage IN ('Prospecting', 'Engaging')
  AND pr.product IS NULL
GROUP BY p.product
ORDER BY open_deals DESC
"""
    )
)

### Setup / lógica / limites

- **Setup:** Lab 01 → este notebook → Run All. Filtro `sales_agent` no widget.
- **Lógica:** score de regras (stage, tamanho de conta, match de produto, win rate do agente, recência). Não é `ORDER BY close_value`.
- **Limites:** não há atividade de e-mail/call no dump; `GTXPro` precisa do replace; 5 agentes sem deal; score não é probabilidade calibrada. Para escalar: app Streamlit + o mesmo SQL.
- Lab 04 testa se campos predizem Won/Lost (AUC ~0,5) — por isso a ferramenta do vendedor é regra + explicação, não o modelo sozinho.

### Painéis gráficos

Ranking, galeria, classificação, segmentação, segmentação de imagem, estatísticas do modelo, matriz de confusão, curva de aprendizado e viés/variância. Calculados só com colunas reais do dump.


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / '03_LAB_Dashboard', HERE.parent / 'lib', HERE.parent.parent / 'lib'):
    if (candidate / 'lab03_panels' / '__init__.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('lab03_panels não encontrado')

from lab03_panels import build_crm_panels
from lab03_panels.render import render_panels

try:
    show = display
except NameError:
    show = print

panels = build_crm_panels()
render_panels(panels, show)
print('secoes', list(panels.panels))
